# Foundation models as summary backbones


_Author: Stefan T. Radev_

Amortized inference tries to solve two problems at the same time: representation learning and inference. A foundation model can be used as a plug-in solution for the former, which leaves us with the inference problem. This notebook demonstrates how to use embeddings from pre-trained foundation models as part of BayesFlow workflows:

![Example workflow with foundation models as summaries](misc/foundation/embedding_workflow.png)

We will do this twice: 
1. TabPFN for nonlinear regression, then 
2. Chronos-2 for a short multivariate time series. 

TabPFN embeddings condition the inference network directly. Chronos embeddings first pass through a learned bottleneck that returns 16 summary statistics. We precompute both sets of embeddings and use **offline** learning, so training does not involve active calls to the frozen encoders.

Finally, we use TabPFN not as a backbone but as the inference network itself.


> Important! An embedding from a pretrained foundation model need not be sufficient for the parameters we care about. We can only learn a posterior conditional on the retained representation, which may be less informative than the full data or specialized summaries. Another approach is to fine-tune parts of the encoder for the estimation task at hand (not explored here). Let's see how this maps onto our BayesFlow terminology:

## Where does the backbone fit?

| Approach | BayesFlow input | What is trained? | Main tradeoff |
|---|---|---|---|
| Offline, frozen encoder | Cached vectors as `inference_conditions` | Inference network | Fast repeated training; a fixed simulation bank and extra storage. |
| Online, frozen encoder | Raw `summary_variables` through a wrapped encoder | Inference network | Simulate during training; encode every batch. |
| Frozen encoder + projection | Cached vectors through a small `summary_network` | Projection and inference network | Adapts the representation cheaply, but cannot recover information already discarded. |
| Partial fine-tuning | Raw `summary_variables` | Selected encoder blocks or adapters, plus inference network | More flexibility; more memory and tuning. |
| Full fine-tuning | Raw `summary_variables` | Encoder and inference network | Most flexible; highest cost and risk of overfitting. |

The TabPFN example uses direct conditioning (first row). The Chronos example uses a projection (third row): cached vectors enter as `summary_variables`, and a small `SummaryNetwork` compresses them before conditioning the inference network.

**Match the backend for online integration.** A PyTorch encoder belongs in a BayesFlow model using the Torch backend. Thus, you should set `KERAS_BACKEND` before importing Keras or BayesFlow. Then, the encoder can become part of a small custom `SummaryNetwork` which simply calls the encoder and optionally passes its outputs through a projection layer. Fine-tuning needs a differentiable forward call; NumPy conversion and high-level embedding helpers often stop gradients. Cached NumPy embeddings can be used with any BayesFlow backend.

A frozen model still needs evaluation mode to disable dropout. For caching, use `eval()` and a no-gradient call. For fine-tuning, use training mode for the parts being optimized and keep the tensor computation intact.

## Choosing an encoder

It is the modeler's task to find the **representation before the prediction head**. Forecasts, class probabilities, and generated text are different outputs and may discard information useful for your simulator. The following are practical starting points, checked in October 2026.

| Data | Models to consider | Where to get the representation |
|---|---|---|
| Tables | [TabPFN-3.5; public v2 for this demo](https://github.com/PriorLabs/TabPFN) | `get_embeddings`; pool the contextual row embeddings. |
| Time series | [Chronos-2](https://huggingface.co/amazon/chronos-2), [Chronos-Bolt tiny](https://huggingface.co/amazon/chronos-bolt-tiny) | `embed` exposes encoder states. Chronos-2 handles multivariate context; Bolt is a small univariate model. |
| Time series | [TimesFM 3.0 and 2.5](https://github.com/google-research/timesfm) | 3.0 is newer; [2.5 has a Transformers base-model API](https://huggingface.co/docs/transformers/model_doc/timesfm2_5) returning hidden states. |
| Time series | [MOMENT](https://github.com/moment-timeseries-foundation-model/moment), [IBM Granite TTM](https://huggingface.co/collections/ibm-granite/granite-time-series) | Embedding task or backbone hidden states; check channel pooling. |
| Images | [DINOv3](https://huggingface.co/docs/transformers/model_doc/dinov3) | Image processor, then the base encoder's CLS token or patch features. |
| Text | [Qwen3-Embedding](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) | Tokenizer and documented sentence embedding/pooling. |
| Audio | [CLAP](https://huggingface.co/docs/transformers/model_doc/clap) | Audio processor and audio features; match the required sample rate. |

Check the checkpoint's license and access requirements. These examples use public checkpoints: TabPFN v2 and the 120-million-parameter Chronos-2.

### A short Hugging Face guide

`from_pretrained` downloads and caches model weights. It does not make every model an interchangeable encoder. Read the model card for preprocessing, masks, normalization, and pooling. For encoder–decoder models, select the encoder; for decoder-only models, select the hidden states before the task head.

For example, the following is an **optional TimesFM alternative**, not needed for this notebook's run. Its base-model API exposes the final patch representation directly. The input here has 128 time steps, a multiple of its 32-step patch length.

```python
from transformers import TimesFm2_5Model

encoder = TimesFm2_5Model.from_pretrained("google/timesfm-2.5-200m-transformers").eval()

with torch.inference_mode():
    output = encoder(past_values=series)  # (datasets, 128)
    embedding = output.last_hidden_state[:, -1]
    # Retain output.loc and output.scale if level or scale carry parameter information.
```

[TimesFM's documented outputs](https://huggingface.co/docs/transformers/model_doc/timesfm2_5) differ from those of a text or image encoder. Do not blindly average every axis: preserve channel identity, exclude padding, and use the model's documented pooling rule. Save the checkpoint version, preprocessing, and pooling choices with your BayesFlow estimator.

## Setup

Restart the kernel before running the notebook. Install BayesFlow from this repository (`python -m pip install -e .` at the repository root), then install the example's extra dependencies:

```python
%pip install "tabpfn==9.2.0" "chronos-forecasting==2.3.2" "transformers==5.4.0" "torch>=2.6"
```

The first run needs internet access to download the two checkpoints. Later runs reuse the model cache. The encoders use CUDA when available, as does BayesFlow's Torch backend. We use one TabPFN ensemble member. Larger simulation banks improve validation but increase the one-time embedding cost.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import keras
import torch
import bayesflow as bf

from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

In [ ]:
rng = np.random.default_rng(2026)

device = "cuda" if torch.cuda.is_available() else "cpu"

plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "figure.dpi": 110,
    "axes.prop_cycle": plt.cycler(color=["#287D8E", "#D58B40", "#596A85"]),
})

## 1. TabPFN: nonlinear regression with unequal noise

We use the model from [Generalized linear models](Bayesian_GLM.ipynb), now with a pretrained encoder. Each dataset has 100 rows and two predictors. We keep the design matrix fixed across simulations and observations. The response has a sinusoidal effect, an interaction, and noise that increases with the second predictor:

$$y_i = b + a\sin(1.5 x_{i1}) + c x_{i1}x_{i2} + 0.5x_{i2} + \epsilon_i,$$
$$\epsilon_i \sim \mathcal N\!\left(0,\,[\exp(\ell)(1+0.4|x_{i2}|)]^2\right).$$

We infer $\theta=(b,a,c,\ell)$, where $\ell=\log\sigma$. Independent normal priors allow positive and negative effects. This is a dataset-level inference problem: TabPFN sees the observed predictors and responses, whereas our inference network (normalizing flow, flow matching, *etc.*) learns the four-parameter posterior.

We train the inference head on **4,096 simulated datasets**, compared with **384,000** (`60 × 100 × 64`) in the GLM notebook with its learned SetTransformer, resulting in about 94 times fewer training simulations.

In [ ]:
x = rng.uniform(-2, 2, size=(100, 2))
basis = np.column_stack([np.ones(100), np.sin(1.5 * x[:, 0]), x[:, 0] * x[:, 1]])
prior_mean = np.array([0.0, 0.0, 0.0, -0.5])
prior_std = np.array([0.7, 0.8, 0.6, 0.3])


def regression_response(theta):
    mean = theta[:, :3] @ basis.T + 0.5 * x[:, 1]
    scale = np.exp(theta[:, 3:4]) * (1 + 0.4 * np.abs(x[:, 1]))
    return (mean + scale * rng.normal(size=mean.shape))


def simulate_regression(n):
    theta = rng.normal(prior_mean, prior_std, size=(n, 4))
    return {"theta": theta, "y": regression_response(theta)}


train_raw, val_raw, test_raw = [simulate_regression(n) for n in (4096, 256, 128)]

### Compute the frozen summaries once

[TabPFN's `fit`](https://github.com/PriorLabs/TabPFN) prepares a dataset as in-context input; it does not optimize the pretrained weights. We reuse one model, call `fit` and `get_embeddings` for each dataset, then average its row embeddings. The public embedding method returns NumPy arrays, which is exactly what we need for offline training.

TabPFN [standardizes each dataset's response](https://github.com/PriorLabs/TabPFN/blob/main/src/tabpfn/regressor.py). We append its mean and standard deviation so that location and scale information remain available. The predictors are fixed here; for varying designs, also consider information removed by predictor preprocessing.

All observed responses are legitimate conditioning data for posterior inference. We therefore use `data_source="train"` for both simulations and observations. This is different from predicting a held-out row's response.

In [ ]:
backbone = TabPFNRegressor.create_default_for_version(
    ModelVersion.V2, 
    n_estimators=1, 
    device=device,
    fit_mode="fit_preprocessors"
)


def tabpfn_summary(responses):
    summaries = []
    for y in responses:
        backbone.fit(x, y)
        # One query row is enough: we request the 100 context (train) embeddings.
        rows = backbone.get_embeddings(x[:1], data_source="train")[0]
        summaries.append(np.r_[rows.mean(axis=0), y.mean(), y.std()])

    return np.asarray(summaries)


train, val, test = [
    {"theta": data["theta"], "embedding": tabpfn_summary(data["y"])}
    for data in (train_raw, val_raw, test_raw)
]

### Train a small inference network

`theta` becomes `inference_variables`; the cached `embedding` becomes `inference_conditions`. We use a two-layer spline coupling flow with small subnets and dropout. There is no summary network, and training only uses the cached embeddings.

BayesFlow standardizes targets and conditions using the training data. We monitor validation loss over a fixed number of epochs. Dropout is active during training, so training loss can be higher than validation loss.

In [ ]:
workflow = bf.BasicWorkflow(
    inference_variables="theta",
    inference_conditions="embedding",
    inference_network=bf.networks.CouplingFlow(
        depth=2, 
        transform="spline",
        subnet_kwargs={"widths": (128, 128), "dropout": 0.3},
    ),
    standardize="all",
    initial_learning_rate=1e-3
)

history = workflow.fit_offline(train, validation_data=val, epochs=80, batch_size=64, verbose=2)

### Loss, parameter recovery, and calibration

BayesFlow's diagnostics show losses, parameter recovery, and calibration on test simulations. Recovery uses posterior medians and central 90% credible intervals. The rank ECDF below each recovery plot checks calibration: systematic departures from zero point to a biased or miscalibrated posterior.

In [ ]:
bf.diagnostics.loss(
    history, 
    smoothing_factor=0,
    val_marker="", 
    figsize=(8, 3.5),
)

samples = workflow.sample(conditions={"embedding": test["embedding"]}, num_samples=500)["theta"]

bf.diagnostics.recovery(
    estimates=samples, 
    targets=test["theta"],
    variable_names=["Intercept", "Sine effect", "Interaction", "Log noise"],
    uncertainty_agg_kwargs={"prob": 0.9}, color="#287D8E",
    num_col=4, 
    figsize=(12, 3.2), 
    markersize=3,
    label_fontsize=11, 
    title_fontsize=12, 
    metric_fontsize=11, 
    tick_fontsize=10,
)

bf.diagnostics.calibration_ecdf(
    estimates=samples, targets=test["theta"],
    variable_names=["Intercept", "Sine effect", "Interaction", "Log noise"],
    difference=True, num_col=4, figsize=(12, 3.2), rank_ecdf_color="#287D8E",
    label_fontsize=11, title_fontsize=12, tick_fontsize=10, legend_fontsize=9,
)

### Use the same encoder on observations

Here we create one synthetic observed dataset so its true parameters are known. For real data, replace `observed_y` with the 100 observed responses on the same design. Only the embedding enters `sample`; the true parameters are used for plotting.

In [ ]:
observed_theta = np.array([[0.3, 1.0, -0.5, -0.6]], dtype="float32")
observed_y = regression_response(observed_theta)
observed_embedding = tabpfn_summary(observed_y)
observed_samples = workflow.sample(
    conditions={"embedding": observed_embedding}, num_samples=2000,
)["theta"][0]

bf.diagnostics.pairs_posterior(
    observed_samples, targets=observed_theta[0],
    variable_names=["Intercept", "Sine effect", "Interaction", "Log noise"],
    post_color="#287D8E", target_color="#D58B40",
    height=2.2, place_legend_below=True,
)

## 2. Chronos-2: infer a full VAR matrix

The same workflow works for sequences. [Chronos-2](https://huggingface.co/amazon/chronos-2) has a public multivariate `embed` method. Its encoder lets the two channels exchange information while keeping independent datasets separate. We use that joint representation to infer both within-channel persistence and cross-channel effects.

Our data are two coupled time series:

$$x_t = A x_{t-1} + 0.4\epsilon_t,\qquad A = \begin{pmatrix}a_{11} & a_{12} \\ a_{21} & a_{22}\end{pmatrix},\qquad \epsilon_t\sim\mathcal N(0,I).$$

We infer all four entries of $A$, with independent $a_{ij}\sim\mathcal U(-0.45,0.45)$. The matrix stays fixed within each dataset. Its maximum absolute row sum is below 0.9, so every prior draw is stable. Each dataset has 128 observations after a short burn-in. We use 4,096 training simulations, 512 validation simulations, and 128 test simulations.

Unlike the univariate Chronos-Bolt encoder, Chronos-2 receives both channels together. We keep channel identity when pooling and retain the means and scales removed by normalization.

In [ ]:
def simulate_var(theta, n_steps=128, burn_in=64):
    theta = np.asarray(theta, dtype=np.float32)
    matrix = theta.reshape(-1, 2, 2)
    state = np.zeros((len(theta), 2), dtype=np.float32)
    series = np.empty((len(theta), 2, n_steps), dtype=np.float32)
    for t in range(n_steps + burn_in):
        state = np.einsum("bij,bj->bi", matrix, state) + 0.4 * rng.normal(size=state.shape)
        if t >= burn_in:
            series[:, :, t - burn_in] = state
    return series


var_names = [r"$a_{11}$", r"$a_{12}$", r"$a_{21}$", r"$a_{22}$"]
var_theta = rng.uniform(-0.45, 0.45, size=(4096 + 512 + 128, 4))
var_series = simulate_var(var_theta)

In [ ]:
from chronos import Chronos2Pipeline

chronos = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map=device,
    dtype=torch.float32,
)
chronos.model.eval();

### Encode once, then train a bottleneck and inference head

The [public `embed` method](https://github.com/amazon-science/chronos-forecasting/blob/main/src/chronos/chronos2/pipeline.py) accepts `(datasets, channels, time)` arrays and handles batching. It returns encoder tokens and normalization statistics on the CPU, without running the forecasting head.

We average the observed-context tokens separately for each channel, excluding the final `[REG]` and masked forecast tokens. Concatenating both channel summaries and their means and scales gives 1,540 features per dataset.

Here we route the cached embeddings to `summary_variables`. A small `SummaryNetwork` compresses them to 16 summary statistics: one 32-unit hidden layer with L2 regularization and dropout, followed by a linear output layer. This limits capacity and discourages overfitting. BayesFlow standardizes the embeddings and trains the bottleneck jointly with the coupling flow; Chronos stays frozen.

The adapter maps all four bounded coefficients to an unconstrained training space and maps posterior draws back to $(-0.45, 0.45)$.

In [ ]:
def embed_var(series):
    tokens, loc_scale = chronos.embed(series, batch_size=64)
    pooled = torch.stack([item[:, :-2].mean(1).flatten() for item in tokens])
    stats = torch.stack([torch.cat(pair, dim=-1).flatten() for pair in loc_scale])
    return torch.cat((pooled, stats), dim=-1).numpy()


var_embedding = embed_var(var_series)
var_train = {"theta": var_theta[:4096], "embedding": var_embedding[:4096]}
var_validation = {"theta": var_theta[4096:4608], "embedding": var_embedding[4096:4608]}
var_test = {"theta": var_theta[4608:], "embedding": var_embedding[4608:]}

In [ ]:
@bf.utils.serialization.serializable("custom")
class EmbeddingBottleneck(bf.networks.SummaryNetwork):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.hidden = keras.layers.Dense(
            32, activation="swish", kernel_regularizer=keras.regularizers.L2(1e-3)
        )
        self.dropout = keras.layers.Dropout(0.5)
        self.summary_stats = keras.layers.Dense(16)

    def call(self, embeddings, training=False, **kwargs):
        hidden = self.hidden(embeddings)
        hidden = self.dropout(hidden, training=training)
        return self.summary_stats(hidden)


var_adapter = (
    bf.Adapter().constrain("theta", lower=-0.45, upper=0.45)
    .rename("theta", "inference_variables").rename("embedding", "summary_variables")
)

var_workflow = bf.BasicWorkflow(
    adapter=var_adapter,
    summary_network=EmbeddingBottleneck(),
    inference_network=bf.networks.CouplingFlow(
        depth=2, 
        transform="spline",
        subnet_kwargs={"widths": (64, 64), "dropout": 0.5},
    ),
    standardize="all",
    initial_learning_rate=1e-3,
)

var_history = var_workflow.fit_offline(
    var_train, validation_data=var_validation, epochs=50, batch_size=64, verbose=2
)

In [ ]:
bf.diagnostics.loss(var_history, smoothing_factor=0, val_marker="", figsize=(8, 3.5))

var_samples = var_workflow.sample(conditions={"embedding": var_test["embedding"]}, num_samples=500)["theta"]

bf.diagnostics.recovery(
    estimates=var_samples, 
    targets=var_test["theta"], 
    variable_names=var_names,
    uncertainty_agg_kwargs={"prob": 0.9}, 
    color="#287D8E",
    num_col=4, 
    figsize=(12, 3.2), 
    markersize=3,
    label_fontsize=11, 
    title_fontsize=12, 
    metric_fontsize=11, 
    tick_fontsize=10
)

bf.diagnostics.calibration_ecdf(
    estimates=var_samples, targets=var_test["theta"], variable_names=var_names,
    difference=True, num_col=4, figsize=(12, 3.2), rank_ecdf_color="#287D8E",
    label_fontsize=11, title_fontsize=12, tick_fontsize=10, legend_fontsize=9,
)

### A new observed trajectory

For an observed dataset, we can run the same frozen encoder once and reuse the trained bottleneck and inference head. Here we simulate one observation at known coefficients so that we can compare its posterior with the generating values. The training, validation, and test datasets above used independent prior draws.

In [ ]:
var_truth = np.array([[0.40, 0.20, -0.15, -0.35]])
var_observed = simulate_var(var_truth)

var_observed_samples = var_workflow.sample(conditions={"embedding": embed_var(var_observed)}, num_samples=2000)
var_observed_samples = var_observed_samples["theta"][0]

fig, axes = plt.subplots(1, 5, figsize=(16, 3.1), layout="constrained")
for channel, color in enumerate(["#287D8E", "#D58B40"]):
    axes[0].plot(var_observed[0, channel], color=color, lw=1.2, label=f"Channel {channel + 1}")

axes[0].set(title="Observed trajectory", xlabel="Time", ylabel="Value")
axes[0].legend(frameon=False, fontsize=9)

for j, label in enumerate(var_names):
    ax = axes[j + 1]
    ax.axvline(var_truth[0, j], color="#D58B40", lw=2, label="Generating value", zorder=3)
    ax.hist(var_observed_samples[:, j], bins=35, density=True, color="#287D8E",
            alpha=0.8, label="Posterior", zorder=2)
    sns.kdeplot(
        x=var_theta[:, j], ax=ax, color="0.55", fill=True, alpha=0.18, linewidth=1,
        cut=0, clip=(-0.45, 0.45), label="Prior", zorder=0,
    )
    ax.set(title=f"Posterior for {label}", xlabel=label, ylabel="Density", xlim=(-0.45, 0.45))

axes[1].legend(frameon=False, fontsize=9)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)

## Taking this to your own simulator

Keep the encoder, preprocessing, and pooling identical for simulations and observations. Cache their outputs when the backbone is frozen; only rerun it for new datasets. Save those choices and any fitted projection alongside the estimator, since the BayesFlow file does not contain an external encoder.

Start with parameter recovery and calibration on test simulations. Compare against a small learned summary network or domain-specific summaries if accuracy matters. A pretrained representation can save training, but its pretraining objective does not guarantee that it retains your simulator's parameters.

## 3. TabPFN as the inference network

So far, a foundation model compressed the data and BayesFlow learned the posterior. We can also let TabPFN estimate the posterior directly, as in [NPE-PFN](https://arxiv.org/abs/2504.17660). TabPFN predicts a full distribution for one target from a table of examples. Our simulations form that table: a parameter is the target and the 100 responses are the features. The four-parameter posterior follows from an autoregressive factorization,

$$p(\theta \mid y) \approx \prod_{j=1}^{4} q(\theta_j \mid \theta_{<j}, y),$$

where each factor is one TabPFN prediction with the simulations as context. Sampling draws $\theta_1$, appends it to the features, draws $\theta_2$, and so on.

Nothing is trained. Simulations are only stored as context. Here, the cost moves to sampling instead as every draw needs four passes through TabPFN with all simulations in context. For larger budgets or data TabPFN will need filtering or summaries, as discussed in the paper. Increasing the number of context rows improves accuracy but increases sampling time.

In [ ]:
posterior_model = TabPFNRegressor.create_default_for_version(
    ModelVersion.V2,
    n_estimators=1,
    device=device,
    # lifts the 1,000-row guard on CPU
    ignore_pretraining_limits=True,
)


def tabpfn_posterior(responses, num_samples, n_context=100):
    features = np.repeat(responses, num_samples, axis=0)
    for j in range(train_raw["theta"].shape[1]):
        # Responses and the preceding parameters predict parameter j.
        posterior_model.fit(
            np.c_[train_raw["y"][:n_context], train_raw["theta"][:n_context, :j]],
            train_raw["theta"][:n_context, j]
        )
        output = posterior_model.predict(features, output_type="full")
        draws = output["criterion"].sample(output["logits"]).cpu().numpy()
        features = np.c_[features, draws]

    return features[:, responses.shape[1]:].reshape(len(responses), num_samples, -1)

### Recovery and calibration without training

We run the same diagnostics on the same test simulations as in the first example, with 100 posterior draws per dataset to keep sampling cheap. However, inference will still be comparatively slow.

In [ ]:
pfn_samples = tabpfn_posterior(test_raw["y"], num_samples=100)

bf.diagnostics.recovery(
    estimates=pfn_samples,
    targets=test_raw["theta"],
    variable_names=["Intercept", "Sine effect", "Interaction", "Log noise"],
    uncertainty_agg_kwargs={"prob": 0.9}, color="#287D8E",
    num_col=4,
    figsize=(12, 3.2),
    markersize=3,
    label_fontsize=11,
    title_fontsize=12,
    metric_fontsize=11,
    tick_fontsize=10,
)

bf.diagnostics.calibration_ecdf(
    estimates=pfn_samples, targets=test_raw["theta"],
    variable_names=["Intercept", "Sine effect", "Interaction", "Log noise"],
    difference=True, num_col=4, figsize=(12, 3.2), rank_ecdf_color="#287D8E",
    label_fontsize=11, title_fontsize=12, tick_fontsize=10, legend_fontsize=9,
)

### The same observation

The observed responses from the first example enter TabPFN directly without an embedding.

In [ ]:
pfn_observed_samples = tabpfn_posterior(observed_y, num_samples=100)[0]

bf.diagnostics.pairs_posterior(
    pfn_observed_samples, targets=observed_theta[0],
    variable_names=["Intercept", "Sine effect", "Interaction", "Log noise"],
    post_color="#287D8E", target_color="#D58B40",
    height=2.2, place_legend_below=True,
)

If TabPFN is the inference network, the same checks apply as with other inference networks. Sampling then calls the foundation model for every draw, so it pays off for small simulation budgets and few observations as it skips training but not for amortization over many datasets.
